## Qualitative Analysis

1. Label validity: Does the original gold label (RQ/IQ) still hold with the swapped context?
2. Coherence: Does the swapped context form a make sense with the target question?
3. Prediction sensitivity: Do Logistic Regression and RoBERTa change their predictions? Does that correspond to a plausible change in interpretation?

In [11]:
# imports

import pandas as pd
import numpy as np

In [12]:
# loading processed data and saved predictions

test = pd.read_json("../data/test_processed.jsonl", lines=True)

lr = pd.read_csv("lr_predictions.csv")
roberta = pd.read_csv("roberta_predictions.csv")

In [13]:
# combining both model predictions into one dataframe

df = lr.merge(
    roberta,
    on="id",
    suffixes=("_lr", "_roberta"),
    validate="one_to_one"
)

# sanity checking size, question-matching, and order
assert len(df) == len(test) == 438
assert (df["question_lr"] == df["question_roberta"]).all()
assert (df["question_lr"].to_numpy() == test["question"].to_numpy()).all()

# adding the contexts
df["original_context"] = test["preceding_turns"].to_numpy()
df["swapped_context"] = test["swapped_context"].to_numpy()

# identifying changed predictions for both models
df["lr_changed"] = df["lr_real"] != df["lr_swapped"]
df["roberta_changed"] = df["roberta_real"] != df["roberta_swapped"]

In [14]:
# choosing samples from specific categories of prediction changes between real and swapped context

label_map = {0: "informational", 1: "rhetorical"}

df["roberta_real_label"] = df["roberta_real"].map(label_map)
df["roberta_swapped_label"] = df["roberta_swapped"].map(label_map)

# defining the five groups
both = df["lr_changed"] & df["roberta_changed"]
same_direction = df["lr_real"] == df["roberta_real_label"]

groups = {
    "both_same": df[both & same_direction],
    "both_opposite": df[both & ~same_direction],
    "lr_only": df[df["lr_changed"] & ~df["roberta_changed"]],
    "roberta_only": df[~df["lr_changed"] & df["roberta_changed"]],
    "neither": df[~df["lr_changed"] & ~df["roberta_changed"]]
}

# getting samples with random seed 42 (2 for each group)
samples = []

for name, group in groups.items():
    selected = group.sample(n=2, random_state=42).copy()
    selected["category"] = name
    samples.append(selected)

qualitative_sample = pd.concat(samples, ignore_index=True)

In [15]:
# exporting annotation sheet for manual analysis

columns = [
    "id", "category", "question_lr", "gold_lr",
    "original_context", "swapped_context",
    "lr_real", "lr_swapped",
    "roberta_real_label", "roberta_swapped_label"
]

qualitative_sample[columns].to_csv(
    "qualitative_analysis.csv",
    index=False
)